In [22]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Multi-Agent_Systems"


In [23]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI


In [40]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# Creating subagents

In [35]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [36]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

# Calling subagents

In [37]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

# Test

In [38]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [39]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='49eb0eb2-221a-4df5-8ec9-76b41b1f3e30'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}, '__gemini_function_call_thought_signatures__': {'186b25ab-4c73-4434-b3a1-03e77261981d': 'CuUBAb4+9vtico9JoJv2cZ6ZTIc6mTr84Zyuj6hN5v0X4lqfN6gHtyWFwngCU6T0nf9sNbPIhRpEWyWogYu2IkHoTfVrRVWuKADlheCXYR7RvCrtLydz06KqFK9qbv/orCgAwBXqcq7pQg6L/pqXwxQ3sySp0lSw6UPhLAuSm9Mp5VaoIKgTHBpgKvM0ocq2YooCdGHdgQOT1V5A4pv96hZgYEbkwzW7LikLYKzw6se3BLzlJIKoqE5iQXa56wyMZHCK2GKHosaF+e27PGy6xUdUhULftb5s2plfEkLVSWfH6oazGVOtvg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cd2a4-2f7e-75a2-bd27-13053a6f899d-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '186b25ab-4c73-4434-b3a1-03e77261981d', 'type': 'tool_